# Chapter 1 · Programs you don't write — Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omar-florez/training-efficient-llms/blob/main/labs/ch01_programs.ipynb)

This notebook reproduces every number in **Chapter 1** of *Efficient Training of LLMs*, live: the gradient-descent run of Table 1.1, the divergence you can cause with one hyperparameter, the backprop-by-hand check against PyTorch autograd, the hardware ladder of Figure 1.5, and the overfitting curves of Figure 1.4. Run cells top to bottom; then break things — the suggestions at the end are the actual lab exercises.

📖 Read the chapter: [Programs you don't write](https://omar-florez.github.io/training-efficient-llms/pdf/ch01_programs.pdf)

In [ ]:
import numpy as np, matplotlib.pyplot as plt

NAVY, BLUE, LIGHT, AMBER, GRAY = "#17406b", "#3f74b8", "#9dbfe4", "#b45309", "#5c5c5c"
plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"white","axes.edgecolor":GRAY,
    "axes.labelcolor":"#1a1a1a","axes.grid":True,"grid.color":"#e3eaf3","grid.linewidth":0.8,
    "axes.spines.top":False,"axes.spines.right":False,"font.size":11,"figure.dpi":110})
print("ready")

## 1 · The whole recipe on two parameters

Three apartments, a straight-line model `rent = w·area + b`, mean-squared-error loss, and the four-line loop: **forward, loss, backward, update.** This is a complete training run — the same one that trains a 70B model, minus 6,999,999,998 parameters.

In [ ]:
x = np.array([50., 80., 120.])          # areas (m^2)
y = np.array([1550., 2350., 3700.])     # rents ($)

w, b, lr = 0.0, 0.0, 5e-5
hist = []
for step in range(2001):
    pred  = w * x + b                    # forward
    err   = pred - y
    loss  = (err ** 2).mean()            # loss
    gw    = 2 * (err * x).mean()         # backward (by hand, for 2 params)
    gb    = 2 * err.mean()
    hist.append((step, loss, w, gw))
    w -= lr * gw                         # update
    b -= lr * gb

print(f"{'step':>6} {'loss':>12} {'w':>8} {'dL/dw':>12}")
for s in (0, 1, 2, 10, 100, 2000):
    st, L, wv, g = hist[s]
    print(f"{st:>6} {L:>12,.0f} {wv:>8.2f} {g:>12,.1f}")
print(f"\nprediction for an unseen 100 m² apartment: ${w*100+b:,.0f}")

In [ ]:
steps = np.array([h[0] for h in hist]); losses = np.array([h[1] for h in hist]); ws = np.array([h[2] for h in hist])

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
a1.semilogy(steps[:200], losses[:200], color=NAVY, lw=2)
a1.set(xlabel="step", ylabel="loss (log)", title="The cliff, then the noise floor")
a1.axhline(losses[-1], color=AMBER, ls="--", lw=1); a1.annotate("noise floor", (120, losses[-1]*1.3), color=AMBER)

wgrid = np.linspace(-5, 66, 300)
Lgrid = [((wv*x - y)**2).mean() for wv in wgrid]
a2.semilogy(wgrid, Lgrid, color=NAVY, lw=2)
a2.semilogy(ws[:12], losses[:12], "o-", color=AMBER, ms=5, lw=1.2, label="first 12 steps")
a2.set(xlabel="w  ($/m²)", ylabel="L(w) (log)", title="Descent on the bowl (Fig. 1.2)")
a2.legend(); plt.tight_layout(); plt.show()

## 2 · Cause the divergence yourself

The chapter's claim: for this problem, any learning rate above **η = 1/mean(x²) ≈ 1.29e-4** makes each step land *farther* from the minimum than the last. Test the claim.

In [ ]:
def run(lr, steps=25):
    w = 0.0; out = []
    for _ in range(steps):
        g = 2*((w*x - y)*x).mean()
        out.append(((w*x - y)**2).mean())
        w -= lr*g
    return np.array(out)

plt.figure(figsize=(8, 4))
for lr_try, c in [(5e-5, NAVY), (1.2e-4, BLUE), (1.35e-4, AMBER), (1.6e-4, "#7a1f1f")]:
    plt.semilogy(run(lr_try), lw=2, color=c, label=f"η = {lr_try:.2e}")
plt.axhline(3418, color=GRAY, ls=":", lw=1)
plt.xlabel("step"); plt.ylabel("loss (log)"); plt.legend()
plt.title("One knob, two regimes: converge below the threshold, explode above it")
plt.show()
print("threshold η* = 1/mean(x²) =", 1/np.mean(x**2))

## 3 · Backprop by hand vs. autograd

Chapter 1.3 walks one forward/backward pass with w=2, b=1, x=2, y=7 and gets gradients **(−8, −4)**. PyTorch must agree to the digit. *(This cell needs PyTorch — preinstalled on Colab.)*

In [ ]:
import torch

w_ = torch.tensor(2.0, requires_grad=True)
b_ = torch.tensor(1.0, requires_grad=True)
x_, y_ = torch.tensor(2.0), torch.tensor(7.0)

loss_ = (w_ * x_ + b_ - y_) ** 2       # forward: the graph is recorded here
loss_.backward()                       # backward: chain rule in reverse

print("loss   :", loss_.item(), "   (hand: 4.0)")
print("dL/dw  :", w_.grad.item(), " (hand: -8.0)")
print("dL/db  :", b_.grad.item(), " (hand: -4.0)")
after = (w_ - 0.1*w_.grad) * x_ + (b_ - 0.1*b_.grad)
print("prediction after one step of lr=0.1:", after.item(), " (target: 7.0)")

## 4 · The hardware ladder (Figure 1.5)

Same operation — a matrix multiply — at four rungs of the stack. Two rungs we *measure right now on this machine*; the GPU rungs are NVIDIA's peak dense BF16 datasheet numbers.

In [ ]:
import time

n = 1024
A = np.random.rand(n, n).astype(np.float32); B = np.random.rand(n, n).astype(np.float32)
A @ B                                                    # warm up
t0 = time.perf_counter();  A @ B;  numpy_gf = 2*n**3 / (time.perf_counter()-t0) / 1e9

n2 = 160
A2 = A[:n2,:n2].tolist(); B2 = B[:n2,:n2].tolist()
t0 = time.perf_counter()
C = [[sum(A2[i][k]*B2[k][j] for k in range(n2)) for j in range(n2)] for i in range(n2)]
py_gf = 2*n2**3 / (time.perf_counter()-t0) / 1e9

rungs  = ["pure Python\nloops", "numpy on\nthis CPU", "A100\n(BF16 peak)", "H100\n(BF16 peak)"]
vals   = [py_gf, numpy_gf, 312_000, 989_000]
plt.figure(figsize=(8, 4))
bars = plt.bar(rungs, vals, color=[LIGHT, BLUE, NAVY, NAVY])
plt.yscale("log"); plt.ylabel("GFLOP/s (log)")
for r, v in zip(bars, vals):
    plt.text(r.get_x()+r.get_width()/2, v*1.25, f"{v:,.2f}" if v<1 else f"{v:,.0f}", ha="center", fontsize=10)
plt.title(f"The same matmul, ~{vals[-1]/vals[0]:,.0f}× apart end to end")
plt.show()

## 5 · Overfitting: watch the curves part ways (Figure 1.4)

Fit polynomials of growing capacity to 20 noisy points. Training error falls forever; validation error turns around — the divergence *is* overfitting.

In [ ]:
rng = np.random.default_rng(0)
xd = np.sort(rng.uniform(0, 1, 30)); yd = np.sin(2*np.pi*xd) + rng.normal(0, 0.25, 30)
xt, yt, xv, yv = xd[::2], yd[::2], xd[1::2], yd[1::2]   # train / validation split

degs = range(1, 13); tr, va = [], []
for d in degs:
    c = np.polyfit(xt, yt, d)
    tr.append(np.mean((np.polyval(c, xt)-yt)**2)); va.append(np.mean((np.polyval(c, xv)-yv)**2))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
a1.semilogy(list(degs), tr, "o-", color=NAVY, label="training loss")
a1.semilogy(list(degs), va, "s--", color=AMBER, label="validation loss")
best = list(degs)[int(np.argmin(va))]
a1.axvline(best, color=GRAY, ls=":"); a1.annotate(f"best generalization: degree {best}", (best+0.2, max(va)/2), fontsize=9)
a1.set(xlabel="model capacity (polynomial degree)", ylabel="MSE (log)"); a1.legend()

xs = np.linspace(0, 1, 200)
a2.plot(xt, yt, "o", color=NAVY, label="train pts")
a2.plot(xs, np.polyval(np.polyfit(xt, yt, best), xs), color=BLUE, lw=2, label=f"degree {best}")
a2.plot(xs, np.polyval(np.polyfit(xt, yt, 12), xs), color=AMBER, lw=1.5, ls="--", label="degree 12 (memorizing)")
a2.set_ylim(-2, 2); a2.legend(); a2.set_title("What the two models actually learned")
plt.tight_layout(); plt.show()

## Break things (the actual lab)

1. In §2, find the divergence threshold *empirically* by bisection — how close is it to `1/mean(x²)`?
2. Normalize the areas (`x/100`) and retrain. What learning rate works now? Why does `b` learn faster? (Chapter 1.6 explains what you'll see.)
3. In §5, add 10× more training points. What happens to the gap between the curves — and to the best degree?
4. In §3, replace the loss with `abs(w*x + b - y)` and rerun autograd. Why does the gradient change the way it does?